# R1–R6 재학습 (11/25, 팀 기준)

팀 결정(11/19·11/20) 기준으로 사다리 모델 R1–R6 × 시드 0·1·2 (18개)를 다시 학습하고 개루프 평가까지 한다.
- 분할: 팀원 분할 (검증 11_4·11_7·11_8, 시험 10_1·11_1·11_5)
- 설정: `GOLF_THUNDERHILL_G` (m 1496.9 kg, 앞 제동 몫 0.75, Fxf = 0.699·구동 − 앞 제동, 경사 채널, 좌우 기울기 0)
- 학습 규칙: 1초(25 스텝) 롤아웃 손실, 최대 300 에폭, 조기 종료 25, 학습률 감소 10, 검증 최저 모델

**준비**: 런타임 → 런타임 유형 변경 → GPU. Google Drive 의 `MyDrive/physics_for_driving/` 에 `golf_inputs_1125.zip` 을 올려 둔다
(D7 CSV 20개 + 11/19 OSM 트랙·경사 지도. 우리 저장소 `colab/golf_inputs_1125.zip`).

중간에 끊기면 처음부터 다시 실행하면 된다. 끝난 모델(.json 있음)은 건너뛴다. 모델·로그·평가는 Drive 에 바로 저장된다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/physics_for_driving'
INPUT_ZIP = f'{DRIVE}/golf_inputs_1125.zip'
MODELS = f'{DRIVE}/models_1125'        # R?_s?_roll25.pt / .json
LOGS = f'{DRIVE}/logs_1125'
RESULTS = f'{DRIVE}/results_1125'      # eval_roll25.md / .json, 학습 곡선
import os
for d in (MODELS, LOGS, RESULTS):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(INPUT_ZIP), f'{INPUT_ZIP} 를 먼저 올릴 것'

In [ ]:
# 공용 저장소 shinyoung 브랜치 (main 아님)
REPO = '/content/physics_for_driving'
if not os.path.exists(REPO):
    !git clone -q -b shinyoung https://github.com/Kxngstonx/physics_for_driving.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
!unzip -q -o {INPUT_ZIP} -d {REPO}
!pip -q install commonroad-vehicle-models==3.0.2
%cd {REPO}

In [ ]:
# 팀 결정대로 도는지 점검 (하나라도 FAIL 이면 학습하지 말 것)
!python experiments/golf/check_train_setup.py

## 학습

GPU 하나에 작업 `PARALLEL` 개씩. 느린 물리 모델(R1–R4)을 앞에 둔다. 진행 상황은 `LOGS/train_<모델>.log`.

In [ ]:
PARALLEL = 3
RUNGS = ['R3', 'R1', 'R2', 'R4', 'R5', 'R6']
SEEDS = [0, 1, 2]
import glob, subprocess, time
from concurrent.futures import ThreadPoolExecutor
jobs = [(r, s) for r in RUNGS for s in SEEDS if not os.path.exists(f'{MODELS}/{r}_s{s}_roll25.json')]
print('남은 작업', len(jobs), jobs, flush=True)

def job(rs):
    r, s = rs
    t0 = time.time()
    with open(f'{LOGS}/train_{r}_s{s}.log', 'w') as log:
        rc = subprocess.run(['python', '-m', 'deep_dynamics.golf.run_train', r, '--seed', str(s), '--roll', '25', '--out', MODELS],
                            stdout=log, stderr=subprocess.STDOUT).returncode
    print(('ok    ' if rc == 0 else 'FAILED') + f' {r} s{s}  {(time.time() - t0) / 60:.1f} 분', flush=True)

with ThreadPoolExecutor(PARALLEL) as ex:
    list(ex.map(job, jobs))
print('끝난 모델', len(glob.glob(f'{MODELS}/R?_s?_roll25.json')), '/ 18')

In [ ]:
# 학습 결과 요약 (검증 최저 손실·에폭·시간)
import json, glob
for p in sorted(glob.glob(f'{MODELS}/R?_s?_roll25.json')):
    d = json.load(open(p))
    print(f"{d['tag']:14s} cfg {d['cfg']}  best val {d['best_val']:.4f} @ {d['best_epoch']:3d}  stop {d['stop_epoch']:3d} ({d['stop_reason']})  {d['sec']/60:5.1f} 분")

## 평가 (팀원 `evaluate.py`)

시험 파일 3개, 0.2 s 간격 시작점, 기록 입력으로 25 스텝(1초) 굴림. 0.04 / 0.2 / 0.48 / 1.0 s 의 vx·vy·r RMSE, 발산 비율, 현재값 유지·직선 외삽 기준선.

In [ ]:
import glob
assert len(glob.glob(f'{MODELS}/R?_s?_roll25.json')) == 18, '18개가 다 끝난 뒤 평가'
!python -m deep_dynamics.golf.evaluate --tag roll25 --models {MODELS} --out {RESULTS} > {LOGS}/eval_roll25.log 2>&1 && echo ok || tail -20 {LOGS}/eval_roll25.log
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS}/eval_roll25.md').read()))

In [ ]:
# 내려받기용 묶음 (모델·평가·로그)
!cd {DRIVE} && zip -q -r ladder_1125_results.zip models_1125 results_1125 logs_1125 && ls -lh ladder_1125_results.zip